# CBBQ Bias Analysis with Jacobian Lens
This notebook performs a rigorous bias analysis on the CBBQ (Chinese Bias Benchmark Dataset) using the Qwen3.5-2B model and a pre-fitted Jacobian Lens.

## Tasks Performed:
1. **Robust Token Handling:** Dynamically maps stereotyped answers based on CBBQ category structures and extracts multiple-choice tokens.
2. **Rank Gap Analysis:** Computes how much higher the stereotyped group is ranked compared to the counter-stereotyped group across all 24 layers.
3. **Context Condition Comparison:** Plots the Rank Gap for Ambiguous vs. Disambiguated prompts.
4. **Question Polarity Comparison:** Plots the Rank Gap for Negative vs. Non-Negative prompts.


## Section 1: Setup & Data Loading


In [ ]:
import sys
import json
import torch
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoModelForCausalLM, AutoTokenizer
sys.path.append("../jacobian-lens")
from jlens import JacobianLens, from_hf

MODEL_NAME = 'Qwen/Qwen3.5-2B'
LENS_REPO = 'neuronpedia/jacobian-lens'
LENS_FILE = 'qwen3.5-2b-pt/jlens/Salesforce-wikitext/Qwen3.5-2B-Base_jacobian_lens.pt'
DATA_DIR = Path('../CBBQ/data')

CATEGORIES = [
    'educational_qualification', 'religion', 'race', 'ethnicity', 'age', 
    'sexual_orientation', 'region', 'gender', 'disability', 'nationality', 
    'disease', 'household_registration', 'SES', 'physical_appearance'
]

print("Initializing Model and Pre-fitted Lens...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model_hf = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
model = from_hf(model_hf, tokenizer)
lens = JacobianLens.from_pretrained(LENS_REPO, filename=LENS_FILE)



## Section 2: Dynamic Token Extraction Logic
We dynamically extract the exact group labels from the `answer_info` and map them to expected token sequences.


In [ ]:
category_stereo_map = {
    'educational_qualification': 'ans1',
    'religion': 'ans0',
    'race': 'ans0',
    'ethnicity': 'ans0',
    'age': 'ans1',
    'sexual_orientation': 'ans0',
    'region': 'ans0',
    'gender': 'ans1',
    'disability': 'ans0',
    'nationality': 'ans0',
    'disease': 'ans0',
    'household_registration': 'ans0',
    'SES': 'ans0',
    'physical_appearance': 'ans1'
}

def extract_targets_from_item(item, cat):
    s_key = category_stereo_map[cat]
    c_key = 'ans1' if s_key == 'ans0' else 'ans0'
    
    # In a multiple-choice format, ans0 is mapped to A, ans1 to B.
    if s_key == 'ans0':
        s_tokens = ['A']
        c_tokens = ['B']
    else:
        s_tokens = ['B']
        c_tokens = ['A']
        
    return s_tokens, c_tokens



## Section 3: Rank Gap Computation Engine


In [ ]:
def get_target_ids(tokenizer, words):
    ids = []
    for w in words:
        t_id = tokenizer.encode(w, add_special_tokens=False)
        if t_id: ids.append(t_id[0])
        t_id_space = tokenizer.encode(" " + w, add_special_tokens=False)
        if t_id_space: ids.append(t_id_space[0])
    return list(set(ids))

def compute_rank_gaps(model, lens, tokenizer, prompts, stereo_targets, counter_targets):
    results = []
    for i, prompt in enumerate(prompts):
        lens_logits, _, _ = lens.apply(model, prompt, positions=[-1], use_jacobian=True)
        
        s_ids = get_target_ids(tokenizer, stereo_targets[i])
        c_ids = get_target_ids(tokenizer, counter_targets[i])
        
        item_gaps = {}
        for layer, logits in lens_logits.items():
            sorted_idx = logits[0].argsort(descending=True).tolist()
            
            s_ranks = [sorted_idx.index(tid) for tid in s_ids if tid < len(sorted_idx)]
            c_ranks = [sorted_idx.index(tid) for tid in c_ids if tid < len(sorted_idx)]
            
            min_s_rank = min(s_ranks) if s_ranks else None
            min_c_rank = min(c_ranks) if c_ranks else None
            
            if min_s_rank is not None and min_c_rank is not None:
                item_gaps[layer] = min_c_rank - min_s_rank
            else:
                item_gaps[layer] = 0
        results.append(item_gaps)
    return results



## Section 4: Dataset Processing Loop
We process the items and pool the rank gaps by context condition and question polarity.


In [ ]:
# Data structures to hold aggregated rank gaps across all categories
gaps_ambig = defaultdict(list)
gaps_disambig = defaultdict(list)
gaps_neg = defaultdict(list)
gaps_nonneg = defaultdict(list)

MAX_ITEMS_PER_CAT = 100 

for category in CATEGORIES:
    print(f"Processing Category: {category}...")
    ambig_path = DATA_DIR / category / "ambiguous" / "ambiguous.json"
    disambig_path = DATA_DIR / category / "disambiguous" / "disambiguous.json"
    
    items = []
    if ambig_path.exists():
        with open(ambig_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            items.extend(data[:MAX_ITEMS_PER_CAT//2])
            
    if disambig_path.exists():
        with open(disambig_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            items.extend(data[:MAX_ITEMS_PER_CAT//2])
    
    prompts = []
    s_targets = []
    c_targets = []
    valid_items = []
    
    for item in items:
        s_t, c_t = extract_targets_from_item(item, category)
        if not s_t or not c_t: continue
        
        prompt = f'{item["context"]} {item["question"]}\n选项：\nA. {item["ans0"]}\nB. {item["ans1"]}\nC. {item["ans2"]}\n答案：'
        prompts.append(prompt)
        s_targets.append(s_t)
        c_targets.append(c_t)
        valid_items.append(item)
        
    if not prompts: continue
    
    gaps_list = compute_rank_gaps(model, lens, tokenizer, prompts, s_targets, c_targets)
    
    for item, gaps in zip(valid_items, gaps_list):
        cond = item['context_condition']
        pol = item['question_polarity']
        
        for layer, gap in gaps.items():
            if cond == 'ambiguous': gaps_ambig[layer].append(gap)
            else: gaps_disambig[layer].append(gap)
                
            if pol == 'neg': gaps_neg[layer].append(gap)
            else: gaps_nonneg[layer].append(gap)



## Section 5: Data Visualization
We plot the Average Rank Gap per layer. A positive value means the model is biased toward the stereotyped response.


In [ ]:
layers = list(range(24))

avg_ambiguous = [np.mean(gaps_ambiguous[l]) if gaps_ambiguous[l] else 0 for l in layers]
avg_disambiguousuous = [np.mean(gaps_disambiguousuous[l]) if gaps_disambiguousuous[l] else 0 for l in layers]

avg_neg = [np.mean(gaps_neg[l]) if gaps_neg[l] else 0 for l in layers]
avg_nonneg = [np.mean(gaps_nonneg[l]) if gaps_nonneg[l] else 0 for l in layers]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Context Condition
ax1.plot(layers, avg_ambiguous, label='Ambiguous', marker='o', color='red')
ax1.plot(layers, avg_disambiguousuous, label='Disambiguousuated', marker='x', color='blue')
ax1.set_title('Rank Gap by Context Condition')
ax1.set_xlabel('Layer')
ax1.set_ylabel('Avg Rank Gap (Counter - Stereo)')
ax1.axhline(0, color='black', linestyle='--')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Question Polarity
ax2.plot(layers, avg_neg, label='Negative Polarity', marker='o', color='purple')
ax2.plot(layers, avg_nonneg, label='Non-Negative Polarity', marker='x', color='green')
ax2.set_title('Rank Gap by Question Polarity')
ax2.set_xlabel('Layer')
ax2.set_ylabel('Avg Rank Gap (Counter - Stereo)')
ax2.axhline(0, color='black', linestyle='--')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



## Section 6: Interactive Slice Visualization & Maps
Explore full token-level slice maps just like in the J-Lens walkthrough.


In [ ]:
import os
import threading
from functools import partial
from http.server import HTTPServer, SimpleHTTPRequestHandler
from jlens.vis import build_page, compute_slice

out_base = Path("bbq_slices")
out_base.mkdir(exist_ok=True)

# Generate a slice for 1 example per category
for category in CATEGORIES:
    items = []
    with open(DATA_DIR / f"{category}.jsonl", "r", encoding="utf-8") as f:
        items.append(json.loads(f.readline().strip()))
        
    for i, item in enumerate(items):
        prompt = item['context'] + " " + item['question']
        slug = f"{category}_{i}"
        
        print(f"Generating slice for {slug}...")
        slice_data = compute_slice(model, lens, prompt, mask_display=True)
        out_dir = out_base / slug
        page, _, _ = build_page(
            slice_data,
            prompt,
            title=f"{category} - Example {i}",
            description=prompt,
            mode="fetch",
            out_dir=out_dir,
        )
        (out_dir / "index.html").write_text(page)

print("All slices saved!")
if "_bbq_httpd" not in globals():
    _handler = partial(SimpleHTTPRequestHandler, directory=os.path.abspath("bbq_slices"))
    _bbq_httpd = HTTPServer(("0.0.0.0", 1111), _handler)
    threading.Thread(target=_bbq_httpd.serve_forever, daemon=True).start()
print(f"Server running! Check -> http://0.0.0.0.1111/ (append the slug, e.g., /Gender_identity_0/)")


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from jlens.vis import notebook_iframe

category_prompts = {}
for category in CATEGORIES:
    with open(DATA_DIR / f"{category}.jsonl", "r", encoding="utf-8") as f:
        item = json.loads(f.readline().strip())
        category_prompts[category] = item["context"] + " " + item["question"]

def show_slice(category):
    prompt = category_prompts[category]
    print(f"Prompt: {prompt}")
    slice_data = compute_slice(model, lens, prompt, layer_stride=2, mask_display=True)
    page, _, _ = build_page(slice_data, prompt, title=f"{category} Bias Slice", description=f"Visualizing intermediate layers for {category}")
    return notebook_iframe(page)

widgets.interact(show_slice, category=CATEGORIES);


## Section 6: Translation for Understanding
To help understand the Chinese CBBQ dataset, we sample a few items and translate them to English.

In [ ]:
import random

print("Translating a few examples to English using Qwen...\n")
sample_items = random.sample(valid_items, min(3, len(valid_items)))

for idx, item in enumerate(sample_items):
    print(f"--- Example {idx+1} ({item['category']}) ---")
    
    # Prompt Qwen for translation
    prompt = f"Translate the following Chinese text to English.\n{item['context']} {item['question']}\nTranslation:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model_hf.generate(**inputs, max_new_tokens=100)
    translation = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    print(f"Context & Question (ZH): {item['context']} {item['question']}")
    print(f"Translated (EN): {translation.strip()}\n")
    print(f"Option A: {item['ans0']}")
    print(f"Option B: {item['ans1']}\n")

